## Script for testing results on healthy data

### Library Importations 

In [1]:
import mne
import numpy as np
import pandas as pd
import os.path as op
from sklearn.metrics import accuracy_score, f1_score
import tensorflow as tf 
import os

## Pre-processing data 

In [ ]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 
subject_exclude = ["SS01AH Data"] 

subject_list=  ['NL01SS','NL02IF','NL03JV','NL04NF','NL05WW','NL06DJ','RL01AN','RL02GC','RL03JG','RL04VF','RL05JP',
              'RL06FM','RL07BR','RL08AE','RL09PC','RL10TV', 'RL12JL','RL13MT','RL14LT','RL15CL',
              'RL17CA','RL19RS','RL20EM','RL21MB','RL22AC','RL23ET','RL24DD']


raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/EDF_healthy_participants"

current_index=0
inter_trigger_length=10
window = 50 
step = 1 

In [ ]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject):
    global raw_path
    global frq

    subject_name = subject
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Expected_Muscle"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_healthy_participants.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]

 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   

    epochs.metadata = trial_info[(trial_info["Subject"] == subject.split(" ")[0]) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))]



    return epochs, df_triggers 

In [ ]:
def make_features_df(subject_epoch,subject):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap_ID",
            "Triggers_Order_Nap", # epochs 
            "Muscle Expected",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "Zygo", # processed EMG signal for epoch 
            "Corr"
        ]
        )   
    
    epoch = 0
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))

        nap_id = (t // 60) + 1
        epoch = (t % 60) + 1

        # fill dataframe 

        features.loc[t] = [
            subject, 
            nap_id,
            epoch,
            subject_epoch.metadata['Expected_Muscle'].iloc[0],
            subject_epoch.metadata.iloc[t]["Nb_Zygo"],
            subject_epoch.metadata.iloc[t]["Nb_Corr"],
            epoch_zygo,
            epoch_corr
        ]
        epoch += 1 

    
    return features

In [ ]:
# extracting features for classification 
i = 0 
excl = 0 # count which subjects had to be excluded 

features_results_mat = []

for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0]
            print(subject)


            if subject not in subject_exclude:
                subject_epoch,_ = pre_process_subjets(subject)
                i += 1
                
                # make features data frame for each subject
                features = make_features_df(subject_epoch,subject)
                features_results_mat.append(features)

           

print(f"{i} trials processed, {excl} trials excluded")

In [ ]:
features_all_healthy = pd.concat(features_results_mat, ignore_index=True) # potentially save to excel and load in 

In [ ]:
features_all_healthy.to_pickle("subj_healthy.pkl")

## Training Single Channel Model

In [4]:
features_all_healthy = pd.read_pickle("subj_healthy.pkl") #un pre-processing if don't already have this data frame 
 
X_zygo = features_all_healthy["Zygo"]
X_corr = features_all_healthy["Corr"]


X_zygo = np.array(X_zygo.tolist())
X_corr = np.array(X_corr.tolist())

y_zygo_contr = features_all_healthy["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr_contr = features_all_healthy["Num_Contractions_Corr"].astype(int).to_numpy()

#y_zygo_dur = features_all_healthy["Duration_zygo"].astype(int).to_numpy()
#y_corr_dur = features_all_healthy["Duration_corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)

y_contractions = np.concatenate((y_zygo_contr,y_corr_contr),axis=0)       
#y_durations = np.concatenate((y_zygo_dur,y_corr_dur),axis=0)       
#y = np.concatenate((y_contractions, y_durations), axis=0)
y = y_contractions

## Test Model 

In [6]:
# loading models 
model_contraction = tf.keras.models.load_model("contraction.keras")

In [7]:
# full training results (test data not seen during cross val)
y_pred_healthy_contr = model_contraction.predict(X)  
y_pred_healthy_contr = np.argmax(y_pred_healthy_contr, axis=1)   

# can aso load and test duration and two head models 

# Calculate accuracy
accuracy_healthy_contr = accuracy_score(y_contractions, y_pred_healthy_contr)   

# Calculate F1 score
f1_healthy_contr = f1_score(y_contractions, y_pred_healthy_contr, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Accuracy :", accuracy_healthy_contr)  
print("F1 Score :", f1_healthy_contr)   


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
Model scores---------------
Accuracy : 0.84
F1 Score : 0.8135232432432432
